# 01 — Immigration AI Data Preparation
Load, clean, analyse, and split the Q&A dataset for fine-tuning Flan-T5 with LoRA.

**New format** — each row has: `user_message`, `assistant_response`, `visa_type`, `document_type`, `category`, `language`, `source`, `document_text`

In [ ]:
!pip install datasets transformers pandas matplotlib seaborn --quiet

In [ ]:
import json
import random
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

# Add project root to path so we can import backend modules
sys.path.insert(0, str(Path('..').resolve()))

random.seed(42)
DATA_DIR = Path('../backend/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load JSONL — first run csv_to_jsonl.py if needed

In [ ]:
# Convert CSV → JSONL if the JSONL is missing or stale
csv_path  = DATA_DIR / 'immigration_qa.csv'
jsonl_path = DATA_DIR / 'immigration_qa.jsonl'

if csv_path.exists():
    import subprocess
    result = subprocess.run(
        ['python', '-m', 'backend.data.csv_to_jsonl'],
        cwd='..', capture_output=True, text=True
    )
    print(result.stdout[-2000:] if result.stdout else '')
    if result.returncode != 0:
        print('Warning:', result.stderr[-500:])

records = []
with open(jsonl_path) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)

# Normalise: support both old (input/output) and new (user_message/assistant_response) keys
if 'input' in df.columns and 'user_message' not in df.columns:
    df = df.rename(columns={'input': 'user_message', 'output': 'assistant_response'})

for col in ['visa_type','document_type','category','language','source','document_text']:
    if col not in df.columns:
        df[col] = 'Any' if col == 'visa_type' else ('None' if col == 'document_type' else ('English' if col == 'language' else ''))

print(f'Total records: {len(df)}')
df.head(2)

## 2. Exploratory Data Analysis

In [ ]:
df['q_len'] = df['user_message'].str.len()
df['a_len'] = df['assistant_response'].str.len()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

axes[0,0].hist(df['q_len'], bins=20, color='steelblue', edgecolor='white')
axes[0,0].set_title('Question Length (chars)')

axes[0,1].hist(df['a_len'], bins=20, color='coral', edgecolor='white')
axes[0,1].set_title('Answer Length (chars)')

df['visa_type'].value_counts().plot(kind='barh', ax=axes[0,2], color='#1565C0')
axes[0,2].set_title('By Visa Type')

df['category'].value_counts().plot(kind='barh', ax=axes[1,0], color='#2E7D32')
axes[1,0].set_title('By Category')

df['language'].value_counts().plot(kind='barh', ax=axes[1,1], color='#6A1B9A')
axes[1,1].set_title('By Language')

df['source'].value_counts().plot(kind='barh', ax=axes[1,2], color='#E65100')
axes[1,2].set_title('By Source')

plt.tight_layout()
plt.show()

print(df[['q_len','a_len']].describe().round(1))

## 3. Format Prompts using the project's format_prompt()

In [ ]:
from backend.ml.dataset import format_prompt

def build_example(row):
    prompt = format_prompt(
        user_message  = row['user_message'],
        document_text = row.get('document_text', ''),
        visa_type     = row.get('visa_type', 'Any'),
        document_type = row.get('document_type', 'None'),
        category      = row.get('category', 'General'),
        language      = row.get('language', 'English'),
    )
    return {'input_text': prompt, 'target_text': row['assistant_response']}

formatted = df.apply(build_example, axis=1, result_type='expand')
df = pd.concat([df, formatted], axis=1)

print('=== Sample prompt ===')
print(df['input_text'].iloc[0])
print('\n=== Target ===')
print(df['target_text'].iloc[0])

## 4. Tokenizer Analysis — verify lengths fit model limits

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')

input_tok  = df['input_text'].apply(lambda x: len(tokenizer.encode(x)))
target_tok = df['target_text'].apply(lambda x: len(tokenizer.encode(x)))

print(f'Input tokens  — max: {input_tok.max():,}, mean: {input_tok.mean():.1f}, p95: {input_tok.quantile(0.95):.0f}')
print(f'Target tokens — max: {target_tok.max():,}, mean: {target_tok.mean():.1f}, p95: {target_tok.quantile(0.95):.0f}')

MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 512
over = (input_tok > MAX_INPUT_LEN).sum()
print(f'\nRows exceeding input limit ({MAX_INPUT_LEN} tokens): {over}')
print(f'Recommended: max_input={MAX_INPUT_LEN}, max_target={MAX_TARGET_LEN}')

## 5. Train / Validation / Test Split (80/10/10)

In [ ]:
dataset = Dataset.from_pandas(df[['input_text', 'target_text']])
splits   = dataset.train_test_split(test_size=0.2, seed=42)
val_test = splits['test'].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    'train':      splits['train'],
    'validation': val_test['train'],
    'test':       val_test['test'],
})

print(dataset_dict)
for split, ds in dataset_dict.items():
    print(f'  {split}: {len(ds)} examples')

## 6. Save Processed Dataset + Build RAG Index

In [ ]:
save_path = DATA_DIR / 'processed_dataset'
dataset_dict.save_to_disk(str(save_path))
print(f'Dataset saved to {save_path}')

for split, ds in dataset_dict.items():
    out = DATA_DIR / f'{split}.jsonl'
    with open(out, 'w') as f:
        for row in ds:
            f.write(json.dumps(row) + '\n')
    print(f'Saved {out.name} ({len(ds)} rows)')

In [ ]:
# Build / rebuild ChromaDB RAG index from the full JSONL
from backend.ml.rag import RAGService

print('Building RAG index...')
rag = RAGService()
count = rag.index_jsonl(jsonl_path)
print(f'RAG index built: {count} documents')

## Summary
- Loaded and normalised Q&A JSONL (supports both old and new field names)
- EDA: breakdown by visa_type, category, language, source
- Formatted prompts using `format_prompt()` (same function used in production)
- Verified token lengths fit within Flan-T5 limits
- Split 80/10/10 and saved to disk
- Built ChromaDB RAG index for immediate use even before training

**Next:** run `02_finetune_lora.ipynb` to train the LoRA adapter